> **Original course-project notebook (2026), kept for reference; superseded by `src/dfer/`.** Outputs are cleared (they contained DFEW frames, which cannot be redistributed). See the README for the final pipeline and results.

#Dataset Sanity Check:

In this notebook, we will verify the integrity of the DFEW dataset by checking:

*   Whether the dataset is loaded correctly.
*   The shape and size of video frames.
*   The distribution of the emotion classes.
*   Basic transformations applied to the videos.

We'll begin by inspecting the dataset setup, loading some sample videos, and ensuring everything is working as expected.

##Imports:

We start by importing the necessary libraries for handling datasets, data processing, and visualizations:

* os, sys, Path: For path management and system operations.
* yaml: For loading configuration files.
* torch: For deep learning functionality.
* pandas: For data manipulation (e.g., loading CSV files).
* matplotlib.pyplot: For visualizing data (e.g., plotting class distributions).

These libraries will help us interact with the dataset and visualize the data we work with.

In [ ]:
import os
import sys
from pathlib import Path
import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt

##Set up Paths and Configuration:

In this section, we define the root paths for the project and load configuration files using the yaml library:

* PROJECT_ROOT: The main directory of the project.
* SRC_DIR: The directory where source code files are stored.
* CONFIG_PATH: Path to the configuration file containing dataset paths (paths.yaml).

The configuration file holds:

* DFEW dataset location.
* Splits location (train/test data CSVs).

In [ ]:
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

CONFIG_PATH = PROJECT_ROOT / "configs" / "paths.yaml"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("CONFIG_PATH:", CONFIG_PATH)

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

DFEW_ROOT = config["dfew_root"]
SPLITS_ROOT = config["splits_root"]

print("DFEW_ROOT:", DFEW_ROOT)
print("SPLITS_ROOT:", SPLITS_ROOT)

##Dataset Initialization:

Here, we initialize the train and validation datasets using the DFEWVideoDataset class:

* Dataset Splits: We use CSV files to manage the training and validation splits for fold 1.
* Transformations: We define the transformation pipeline, which resizes images to 224x224, converts them to tensors, and applies normalization based on standard ImageNet values.
* Frames per sample: We will use 16 frames per video clip to ensure uniformity in the input size.

This section loads the data and prints the sizes of the training and validation sets to confirm that the datasets are correctly initialized.

In [ ]:
from dataset import DFEWVideoDataset
from torchvision import transforms

FOLD = 1
TRAIN_CSV = os.path.join(SPLITS_ROOT, "train(single-labeled)", f"set_{FOLD}.csv")
VAL_CSV   = os.path.join(SPLITS_ROOT, "test(single-labeled)",  f"set_{FOLD}.csv")

NUM_FRAMES = 16
IMG_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225))
])

train_ds = DFEWVideoDataset(
    root_dir=DFEW_ROOT,
    split_csv=TRAIN_CSV,
    transform=transform,
    num_frames=NUM_FRAMES
)

val_ds = DFEWVideoDataset(
    root_dir=DFEW_ROOT,
    split_csv=VAL_CSV,
    transform=transform,
    num_frames=NUM_FRAMES
)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

Inspecting a Video Sample:

We inspect one video sample from the training dataset to verify the following:

* Video shape: We expect the video to have the shape (C, T, H, W), where:

    * C: Number of channels (3 for RGB),
    * T: Number of frames (16 in our case),
    * H, W: Height and width (224x224 pixels).

* Label: The emotion class label corresponding to this video.

This will ensure the dataset is loaded correctly and that the frames are in the expected format.

In [ ]:
video, label = train_ds[0]

print("Video shape:", video.shape)   # expected (C,T,H,W)
print("Label:", label)
print("Frames per sample:", NUM_FRAMES)
print("Image size:", IMG_SIZE, "x", IMG_SIZE)

##Emotion Class Names

In [ ]:
CLASS_NAMES = [
    "Happy",
    "Sad",
    "Neutral",
    "Angry",
    "Surprise",
    "Disgust",
    "Fear"
]

In [ ]:
# Collect one example per class
examples = {}

for idx in range(len(train_ds)):
    video, label = train_ds[idx]
    if label not in examples:
        examples[label] = video
    if len(examples) == 7:
        break

fig, axs = plt.subplots(7, 4, figsize=(12, 18))

mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

for row, (label, video) in enumerate(sorted(examples.items())):

    # (C,T,H,W) -> (T,C,H,W)
    if video.shape[0] == 3:
        video = video.permute(1,0,2,3)

    # De-normalize
    video = video * std + mean
    video = video.clamp(0,1)

    for col in range(4):
        frame = video[col].permute(1,2,0).cpu()
        axs[row, col].imshow(frame)
        axs[row, col].axis("off")

    axs[row, 0].set_title(CLASS_NAMES[label], fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
train_labels = df_train["label"] - 1
train_counts = train_labels.value_counts().sort_index()

print("Train class distribution:")
for i in range(len(CLASS_NAMES)):
    print(f"{CLASS_NAMES[i]}: {train_counts[i]}")

plt.figure(figsize=(8,5))
plt.bar(CLASS_NAMES, train_counts)
plt.xticks(rotation=45)
plt.ylabel("Number of samples")
plt.title("DFEW Train Class Distribution")
plt.grid(axis='y')
plt.show()

In [ ]:
df_val = pd.read_csv(VAL_CSV)
val_labels = df_val["label"] - 1
val_counts = val_labels.value_counts().sort_index()

print("Validation class distribution:")
for i in range(len(CLASS_NAMES)):
    print(f"{CLASS_NAMES[i]}: {val_counts[i]}")

plt.figure(figsize=(8,5))
plt.bar(CLASS_NAMES, val_counts)
plt.xticks(rotation=45)
plt.ylabel("Number of samples")
plt.title("DFEW Validation Class Distribution")
plt.grid(axis='y')
plt.show()

In [ ]:
dist_table = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Train Count": train_counts.values,
    "Val Count": val_counts.values
})

dist_table

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

fig.savefig(OUTPUT_DIR / "dfew_class_examples.png", dpi=300)
print("Saved class visualization to outputs/")

##Conclusion

In this notebook, we successfully verified the integrity of the DFEW dataset. We ensured:

* The dataset loaded correctly,
* Video frames have the expected shape and size,
* The emotion class distribution is as expected.

Now, the dataset is ready for further processing and model training.